# Projekt: Auswirkungen von Wetterereignissen auf die urbane Mobilität
### Team: Aron Milojevic, Mateusz Pacyga und Aja Pal.

## 1. Einleitung & Story
In diesem Projekt analysieren wir, wie extremes Wetter (Regen, Schnee) das Buchungsverhalten von Taxis in New York City beeinflusst. 
Wir nutzen dafür Millionen von Datensätzen der NYC TLC und historische Wetterdaten der NOAA.

## 2. Big Data Kriterien (Theoretische Betrachtung)

Gemäß der Projektspezifikation analysieren wir unser Vorhaben anhand der 5 Vs und der 4 Ebenen der Datenverarbeitung.

### Die 5 Vs unseres Projekts
* **Volume:** Wir arbeiten mit dem NYC TLC Datensatz, der Millionen von einzelnen Fahrten enthält und somit ein hohes Datenvolumen aufweist.
* **Velocity:** In der realen Welt entstehen Taxi-Buchungsdaten kontinuierlich im Sekundentakt (Real-time).
* **Variety:** Wir kombinieren zwei unterschiedliche Datentypen: Semistrukturierte Fahrtdaten (JSON/CSV) und strukturierte Wetterdaten (NOAA).
* **Veracity:** Die Datenqualität muss bereinigt werden (Data Cleaning), um Ausreißer wie unmögliche Fahrpreise oder Zeitstempel zu entfernen.
* **Value:** Die Verknüpfung zeigt den wirtschaftlichen Wert auf, indem sie Vorhersagen über die Nachfrage bei Schlechtwetter ermöglicht.

### Die 4 Ebenen der Datenverarbeitung
1. **Data Source:** Nutzung von Open Data Quellen (Kaggle/NYC Open Data & NOAA).
2. **Data Storage:** Speicherung der Rohdaten in einer MongoDB (NoSQL), um flexibel auf Attribute zuzugreifen.
3. **Data Analysis:** Durchführung einer MapReduce-Berechnung zur Aggregation der Fahrten pro Stunde bei Regen/Sonne.
4. **Data Output:** Visualisierung der Ergebnisse in Form von Diagrammen direkt in diesem Jupyter Notebook.

## 3. Infrastruktur & Architektur

Um die Multiuser-Fähigkeit und eine konsistente Arbeitsumgebung zu garantieren, nutzen wir folgende Infrastruktur:

* **Docker:** Kapselung der Dienste in Containern, damit alle Teammitglieder (Aron, Mateusz, Ajay) die identische Umgebung nutzen.
* **MongoDB:** Als NoSQL-Datenbank für die flexible Speicherung der Taxi-Fahrten und Wetterdaten.
* **JupyterLab:** Zentrale Plattform für die Datenanalyse, Dokumentation und Code-Ausführung.
* **Git/GitHub:** Versionskontrolle und gemeinsames Filesharing des Codes zur Ermöglichung einer parallelen Zusammenarbeit.

### Architektur-Diagramm
Das System besteht aus zwei Haupt-Containern (Jupyter und MongoDB), die über ein internes Docker-Netzwerk miteinander kommunizieren. Die Daten werden von lokalen CSV- und Parquet-Dateien eingelesen und in die NoSQL-Datenbank 

![Architektur](architektur.png)überführt.ter zeigt]

## 4. Implementierung und Datenimport

Nachdem die Infrastruktur mittels Docker bereitgestellt wurde, erfolgt nun die technische Umsetzung. In diesem Schritt installieren wir die notwendigen Schnittstellen und laden die Datensätze in unsere NoSQL-Datenbank.

### 4.1 Vorbereitung der Python-Umgebung
Wir installieren `pymongo` für die Datenbankkommunikation und `pyarrow`, um das speichereffiziente Parquet-Format der Taxi-Daten verarbeiten zu können.

In [1]:
!pip install pymongo pyarrow



### 4.2 Aufbau der Datenbankverbindung und ETL-Prozess
Der folgende Code stellt die Verbindung zum MongoDB-Container her und führt den Import der CSV- (Wetter) und Parquet-Dateien (Taxi) durch. Dabei werden die Daten in Dokumente umgewandelt und in Kollektionen gespeichert.

In [2]:
import pandas as pd
from pymongo import MongoClient

# 1. Verbindung zur MongoDB aufbauen (läuft in eurem Docker Container)
# 'mongodb' ist der Name des Services aus eurer docker-compose.yml [cite: 88]
client = MongoClient('mongodb://mongodb:27017/')
db = client['nyc_taxi_weather']

print("Verbindung zur MongoDB steht!")

# 2. Wetter-Daten einlesen und speichern
# WICHTIG: Achte darauf, dass der Dateiname exakt so geschrieben ist wie links in deiner Liste!
weather_file = 'New York, NY, United Stat... 2023-01-01 to 2023-01-31.csv'
weather_df = pd.read_csv(weather_file)

weather_collection = db['weather']
# Wir wandeln die Tabelle in eine Liste von Dokumenten (JSON-Stil) um [cite: 86]
weather_collection.insert_many(weather_df.to_dict('records'))
print(f"Erfolg: {len(weather_df)} Wetter-Datensätze in MongoDB gespeichert!")

# 3. Taxi-Daten einlesen und speichern (Parquet Format)
taxi_file = 'yellow_tripdata_2023-01.parquet'
taxi_df = pd.read_parquet(taxi_file)

# Da Taxi-Daten Millionen Zeilen haben, nehmen wir für den Anfang eine Stichprobe,
# damit der Arbeitsspeicher nicht überläuft (ca. 100.000 Fahrten)
taxi_sample = taxi_df.head(100000)

taxi_collection = db['trips']
taxi_collection.insert_many(taxi_sample.to_dict('records'))
print(f"Erfolg: {len(taxi_sample)} Taxi-Fahrten in MongoDB gespeichert!")

Verbindung zur MongoDB steht!
Erfolg: 744 Wetter-Datensätze in MongoDB gespeichert!
Erfolg: 100000 Taxi-Fahrten in MongoDB gespeichert!
